# Packages and file system management

In [ ]:
import sys

##################################################################################################
# List of packages to check and install if needed
packages = [
    'qiskit',
    'qiskit-machine-learning',
    'qiskit-aer',
    'qiskit-aer-gpu-cu11',
    'qiskit-ibm-runtime',
    'pylatexenc',
]

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"{package} is already installed\n")
    except ImportError:
        print(f"{package} not found, installing...\n")
        !{sys.executable} -m pip install {package}
        print(f"{package} installed successfully\n")

##################################################################################################
# Other required imports
import numpy as np
import math
import matplotlib.pyplot as plt
import random

from torch import no_grad
from torch.nn import CrossEntropyLoss

from qiskit.circuit.library import zz_feature_map, efficient_su2

from pathlib import Path

In [ ]:
from pathlib import Path

##################################################################################################
# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab\n")
except:
    IN_COLAB = False
    print("Running locally\n")

##################################################################################################
# Setup paths
try:
    # Check if already configured
    print(f"Already configured - using: {PROJECT_DIR}")
    print(f"Results file: {RESULTS_FILE}\n")
except NameError:
    # First run - need to configure
    if IN_COLAB:
        user_input = input("Use Google Drive to access modules and store results? (y/n): ").lower().strip()
        while user_input not in ["y", "n"]:
            print("Invalid input. Please enter 'y' or 'n'.")
            user_input = input("Use Google Drive to access modules and store results? (y/n): ").lower().strip()

        if user_input == "y":
            # Use Drive (project + results)
            from google.colab import drive
            drive.mount('/content/drive')
            PROJECT_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Benchmarking_hybrid_Quantum-Classical_CNNs')
            RESULTS_BASE = PROJECT_DIR
            print()
            print(f"Using Google Drive: {PROJECT_DIR}")
        else:
            # Use temporary Colab storage for everything
            PROJECT_DIR = Path.cwd()
            RESULTS_BASE = PROJECT_DIR
            print()
            print(f"Using temporary storage: {PROJECT_DIR}")
            print("All data will be deleted when session ends\n")
            print("Make sure 'src/' folder is uploaded here!")

            # Check if src/ exists
            if not (PROJECT_DIR / 'src').exists():
                print()
                print("ERROR: 'src/' folder not found!")
                print("   Upload it from GitHub or from local before continuing.\n")
                raise FileNotFoundError("src/ folder not found")

    else:   # for local runs
        PROJECT_DIR = Path.cwd()
        RESULTS_BASE = PROJECT_DIR
        print(f"Project directory: {PROJECT_DIR}")

    # Add project to sys.path
    if str(PROJECT_DIR) not in sys.path:
        sys.path.insert(0, str(PROJECT_DIR))

    ##################################################################################################
    # Create directories
    RESULTS_DIR = RESULTS_BASE / 'results'
    CHECKPOINTS_DIR = RESULTS_BASE / 'checkpoints'
    DATA_DIR = RESULTS_BASE / 'datasets'

    for directory in [RESULTS_DIR, CHECKPOINTS_DIR, DATA_DIR]:
        directory.mkdir(parents=True, exist_ok=True)

    print(f"Results directory: {RESULTS_DIR}")
    RESULTS_FILE = RESULTS_DIR / 'benchmark_results.json'
    print(f"\nResults file: {RESULTS_FILE}\n")

# User defined parameters and GPU setup

In [ ]:
CONFIG = {
    "classical_device": "GPU",   # either "GPU" or "CPU"
    "batch_size": 35,
    "images_per_class": 500,
    "employ_quantum_layer": False,   # either True or False
    "optimization": {
        "optimizer": "Adam",   # can be any optimizer class in torch.optim module ("Adam", "SGD", "RMSprop", "LBFGS" etc.)
        "learning_rate": 0.0023,
        "num_epochs": 120,   # for first-order optimizers. Automatically transferred to max_iter, then set to 1 for second-order ones ("LBFGS")
    },
    "quantum_NN": {
        "feature_map": zz_feature_map(feature_dimension = 4),   # can be either a pre-defined circuit or a personalized one
        "ansatz": efficient_su2(num_qubits = 4, reps = 1),   # can be either a pre-defined circuit or a personalized one
        "execution_mode": "exact_simulator"   # either "exact_simulator", "noisy_simulator", "quantum_hardware"
    },
}

##################################################################################################
from src.utils.config_utils import validate_config
from src.utils.device_setup import setup_devices

# Validate CONFIG
validate_config(CONFIG)

# Setup execution devices
torch_device, qiskit_device = setup_devices(CONFIG)

# Load and pre-process dataset, create, train and evaluate model

## Data loading and pre-processing


In [ ]:
# Load and prepare dataset
from src.utils.data_utils import load_dataset

X_train, X_test, train_loader, test_loader = load_dataset(
    data_dir = DATA_DIR,
    samples_per_class = CONFIG["images_per_class"],
    batch_size = CONFIG["batch_size"]
)

The DataLoader object has the following structure:
- dataloader - Type: DataLoader. Lenght: number of bathces (number of samples/```batch_size```)
  - batch - Type: list. Lenght: 2
    - datapoints/images tensor - Type: torch.Tensor. Rank: 4. Dimensions: [```batch_size```, number of channels, height, width] (here [```batch_size```, 3, 32, 32])
      - number of datapoints in the batch (```batch_size```)
      - number of color channels (here 3: RGB)
      - height of the datapoints in pixel (here 32)
      - width of the datapoints in pixel (here 32)
    - labels of the datapoints in the batch - Type: torch.Tensor. Rank: 1. Dimension: ```batch_size``` (except eventually last batch)

In [ ]:
##################################################################################################
# Display some images from the train dataset as a sanity check
n_images_to_show = 13
max_n_samples_per_batch = 5

n_samples_per_batch = min(max_n_samples_per_batch, CONFIG["batch_size"])
n_batches = math.ceil(n_images_to_show/n_samples_per_batch)
iterator = iter(train_loader)   # create an iterator object pointing at batches in the train dataloader

for k in range(n_batches):
  print("-" * 80)
  print(f"Batch {k+1}:")

  batch = next(iterator)   # move the iterator forward by 1 (initially pre-start --> 0) and extract data from the batch it is pointing at
  images = batch[0]   # all the images in the current batch
  labels = batch[1]   # labels of the images in the current batch

  current_n_images = min(n_samples_per_batch, n_images_to_show - k * n_samples_per_batch)
  print(f"Current n images: {current_n_images}")
  fig, axes = plt.subplots(nrows = 1, ncols = current_n_images, figsize = (current_n_images * 2, 3))
  if current_n_images == 1:
    axes = [axes]

  for i in range(current_n_images):
    color_channels = {0: 'Reds', 1: 'Greens', 2: 'Blues'}
    channel = random.randint(0, 2)

    image_to_plot = images[i]

    axes[i].imshow(image_to_plot[channel].numpy().squeeze(), cmap = color_channels[channel])   # transform the image into a numpy, eliminate eventual useless 1-dim dimensions (squeeze), and show it
    axes[i].set_xticks([])
    axes[i].set_yticks([])
    axes[i].set_title(f"Label: {X_train.classes[labels[i]]}")

  plt.show()

## Define Quantum Neural Network and hybrid model

### Quantum Neural Network construction

In [ ]:
from src.utils.quantum_utils import setup_quantum_circuit, setup_estimator, create_qnn

##################################################################################################
# Setup circuit and observables
qc, feature_map_params, ansatz_params, observables = setup_quantum_circuit(
    feature_map = CONFIG["quantum_NN"]["feature_map"],
    ansatz = CONFIG["quantum_NN"]["ansatz"]
)
display(qc.draw(output="mpl", fold=-1))

##################################################################################################
# Setup estimator
estimator, pass_manager = setup_estimator(
    execution_mode=CONFIG["quantum_NN"]["execution_mode"],
    qiskit_device=qiskit_device
)

##################################################################################################
# Create QNN
qnn = create_qnn(qc, feature_map_params, ansatz_params, estimator, pass_manager, observables)

### Implement hybrid Neural Network and initialize model

<center><b>Neural Network Architecture</b></center>

========================================================================================================
**Input Format:** Rank-4 batch tensors with dimensions `[batch_size, num_channels, image_height, image_width]`
- For CIFAR-10: `num_channels = 3` (RGB), `image_height = 32`, `image_width = 32`

========================================================================================================
**Feature Extraction Layers**

1. **Convolutional Layer #1:** Looks for 64 patterns (output channels) in the input image using a 5×5 filter window
   - Output: 28×28 feature maps per channel

2. **Max Pool #1:** Uses a 2×2 filter window (keeps only the most relevant feature in each 2×2 grid)
   - Output: 14×14 feature maps per channel

3. **Batch Normalization #1:** Normalizes activations to stabilize training

4. **ReLU #1:** Nonlinear activation function

5. **Convolutional Layer #2:** Takes 64 input channels and looks for 128 patterns (output channels) using a 3×3 filter window
   - Output: 12×12 feature maps per channel

6. **Max Pool #2:** Uses a 2×2 filter window
   - Output: 6×6 feature maps per channel

7. **Batch Normalization #2:** Normalizes activations to stabilize training

8. **ReLU #2:** Nonlinear activation function

9. **Convolutional Layer #3:** Takes 128 input channels and looks for 256 patterns (output channels) using a 3×3 filter window
   - Output: 4×4 feature maps per channel

10. **Max Pool #3:** Uses a 2×2 filter window
    - Output: 2×2 feature maps per channel

11. **Batch Normalization #3:** Normalizes activations to stabilize training

12. **ReLU #3:** Nonlinear activation function

13. **Dropout:** Randomly turns off 30% of neurons during training to prevent overfitting

14. **Flatten:** Reshapes the data from 256 channels of 2×2 images into a `[batch_size, 1024]` array (1024 = 256×2×2)

15. **Linear Layer #1:** Reduces dimensionality from 1024 neurons to 256

16. **ReLU #4:** Nonlinear activation function

17. **Linear Layer #2:** Reduces dimensionality from 256 neurons to 128

18. **ReLU #5:** Nonlinear activation function

========================================================================================================
**Classification Head**

**If `employ_quantum_layer = True` - Path A:**

19. **Linear Layer #3:** From 128 neurons to `n_features` (number of quantum circuit parameters)

20. **Quantum NN Integration:** Takes `n_features` input parameters and returns `n_qubits` outputs in range [-1, 1] (measured expectation value of Z for each qubit)

21. **Linear Layer #4:** From `n_qubits` neurons to `n_classes` (number of dataset classes)

**If `employ_quantum_layer = False` - Path B:**

19. **Linear Layer #3:** From 128 neurons to 4

20. **GELU Activation:** Gaussian Error Linear Unit (smooth nonlinearity)

21. **Linear Layer #4:** From 4 neurons to `n_classes` (number of dataset classes)

**Output:** Rank-2 tensor `[batch_size, n_classes]` with logits (one value per class for each image in the batch)

In [ ]:
from src.models.hybrid_model import QuantumHybridNet

# Initialize model
model = QuantumHybridNet(config = CONFIG, qnn = qnn, n_classes = len(X_train.classes))
model = model.to(torch_device)

print(model)

## Train model

In [ ]:
from src.training.trainer import train_model

##################################################################################################
# Perform training (optimizer enters here)
loss_func = CrossEntropyLoss()

train_loss_list, training_time = train_model(
    model = model,
    train_loader = train_loader,
    config_dict = CONFIG,
    device = torch_device,
    loss_func = loss_func
  )

##################################################################################################
# Plot loss convergence
plt.plot(train_loss_list)
plt.title("Hybrid NN Training Convergence")
plt.xlabel("Epochs")
plt.ylabel("Cross Entropy Loss")
plt.show()

## Model evaluation and result manager

In [ ]:
from src.training.evaluation import evaluate_model
from src.utils.results_manager import save_results

# Evaluate model on test set
metrics_dict = evaluate_model(
    model = model,
    test_loader = test_loader,
    loss_func = loss_func,
    device = torch_device,
    X_test = X_test
)

# Save results to file
save_results(
    CONFIG,
    metrics_dict,
    training_time,
    RESULTS_FILE,
    torch_device,
    model_name = model.__class__.__name__
)

# Results overview

In [ ]:
from src.utils.results_manager import load_and_compare_results

# Show comparison with previous runs
load_and_compare_results(RESULTS_FILE)

In [ ]:
from src.utils.visualization import plot_results_comparison

# Visualize results comparison
plot_results_comparison(RESULTS_FILE)